In [ ]:
import cv2  # Importation de la bibliothèque OpenCV pour le traitement d'images et la capture vidéo
import mediapipe as mp  # Importation de Mediapipe pour la détection de la pose humaine
import numpy as np  # Importation de NumPy pour les opérations sur les tableaux et calculs mathématiques
import time  # Importation de la bibliothèque time pour gérer les délais dans le traitement vidéo
import pygame  # Importation de la bibliothèque Pygame pour créer des jeux vidéo et des applications multimédia.
from IPython.display import display, clear_output  # Importation des fonctions d'affichage dans Jupyter Notebook
from PIL import Image  # Importation de PIL (Python Imaging Library) pour manipuler les images dans Jupyter

In [ ]:
import pygame  # Importation de la bibliothèque Pygame pour créer des jeux vidéo et des applications multimédia.
import numpy as np  # Importation de la bibliothèque NumPy pour les calculs mathématiques avancés et la gestion de tableaux.


# Initialiser pygame pour le son
pygame.mixer.init(frequency=44100, size=-16, channels=2)

# Fréquence du son en Hz (par exemple, un bip de 1000 Hz)
freq = 1000
duration = 500  # Durée en millisecondes
sample_rate = 44100  # Nombre d'échantillons par seconde
t = np.linspace(0, duration / 1000, int(sample_rate * duration / 1000), endpoint=False)
sound_array = np.sin(2 * np.pi * freq * t)

# Convertir en format audio signé 16 bits et le rendre stéréo
sound_array = np.array([sound_array, sound_array]).T  # Crée une version stéréo
sound_array = (sound_array * 32767).astype(np.int16)  # Convertir en 16 bits signé

# S'assurer que le tableau est C-contiguous
sound_array = np.ascontiguousarray(sound_array)

# Créer le son avec pygame.sndarray.make_sound
sound = pygame.sndarray.make_sound(sound_array)

# Jouer le son
sound.play()

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import pygame
from IPython.display import display, clear_output
from PIL import Image

# ---------- Réglages ----------
CAMERA_INDEX = 0       # ta webcam (confirmée par --list)
GRACE_S = 0.7          # secondes sans détection tolérées avant d'arrêter le chrono

# ---------- Son (un bip) ----------
pygame.mixer.init(frequency=44100, size=-16, channels=2)
freq, duration, sample_rate = 1000, 500, 44100
t = np.linspace(0, duration / 1000, int(sample_rate * duration / 1000), endpoint=False)
sound_array = np.sin(2 * np.pi * freq * t)
sound_array = np.array([sound_array, sound_array]).T
sound_array = np.ascontiguousarray((sound_array * 32767).astype(np.int16))
sound = pygame.sndarray.make_sound(sound_array)

# ---------- MediaPipe ----------
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, model_complexity=1)
mp_drawing = mp.solutions.drawing_utils


def get_angle(a, b, c):
    """Angle en degrés au point b (épaule - hanche - cheville)."""
    a, b, c = np.array(a[:2]), np.array(b[:2]), np.array(c[:2])
    ba, bc = a - b, c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))


def detect_plank(landmarks, width, height):
    """True si la personne est en position de planche."""
    if not landmarks:
        return False

    left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
    left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
    left_ankle = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value]
    left_elbow = landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value]

    right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
    right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value]
    right_ankle = landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value]
    right_elbow = landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value]

    left_angle = get_angle(left_shoulder, left_hip, left_ankle)
    right_angle = get_angle(right_shoulder, right_hip, right_ankle)
    hip_height_diff = abs(left_hip[1] - right_hip[1])
    elbow_height = (left_elbow[1] + right_elbow[1]) / 2

    return (
        170 <= left_angle <= 180
        and 170 <= right_angle <= 180
        and hip_height_diff < 20
        and elbow_height > left_hip[1]
    )


# ---------- Caméra ----------
cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    cap.release()
    raise RuntimeError("Impossible d'ouvrir la caméra (essaie un autre CAMERA_INDEX)")

# ---------- Variables du chronomètre ----------
plank_start_time = None
last_seen = 0.0
elapsed_time = 0.0
plank_count = 0
plank_times = []
is_planking = False

# ---------- Boucle principale ----------
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Erreur : Impossible de capturer l'image")
            break

        height, width, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb_frame)

        landmarks = {}
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            for id, landmark in enumerate(results.pose_landmarks.landmark):
                landmarks[id] = (int(landmark.x * width), int(landmark.y * height), landmark.z)

        is_plank = detect_plank(landmarks, width, height)

        # --- Chronomètre avec tolérance ---
        now = time.time()
        if is_plank:
            last_seen = now
            if not is_planking:
                plank_start_time = now
                plank_count += 1
                is_planking = True
                sound.play()  # un seul bip, au début de la planche
            elapsed_time = now - plank_start_time
        elif is_planking and now - last_seen > GRACE_S:
            plank_times.append(last_seen - plank_start_time)
            print(f"Fin de la planche {plank_count} : {plank_times[-1]:.2f} secondes")
            is_planking = False

        # --- Affichage ---
        text = f"Planche {plank_count}: {elapsed_time:.2f}s" if is_planking else "Pas de planche"
        color = (0, 255, 0) if is_planking else (0, 0, 255)
        cv2.putText(frame, text, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        clear_output(wait=True)
        display(Image.fromarray(frame_rgb))        # affichage dans Jupyter
        cv2.imshow('Detection de pose - Planche', frame)  # fenêtre OpenCV

        time.sleep(0.05)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except KeyboardInterrupt:
    print("Interruption manuelle. Fermeture de la caméra.")

finally:
    # Si une planche est encore en cours, on l'enregistre
    if is_planking:
        plank_times.append(last_seen - plank_start_time)
    cap.release()
    cv2.destroyAllWindows()

print("\nTemps des planches détectées :")
for i, d in enumerate(plank_times, start=1):
    print(f"Planche {i}: {d:.2f} secondes")